# Assignment 03 – Cleaning, Charts and Data Ethics
**Domain:** Cybersecurity – Clinic Security Monitoring  
**Dataset:** UCI PhiUSIIL Phishing URL dataset (same dataset as Assignment 02).

This notebook documents missing-value decisions, outlier inspection, type checks, five charts, findings, and dataset ethics.

## 1. Load the dataset and inspect data quality
The UCI dataset is loaded again so this notebook can run independently. The raw data is retained in `df_raw` and a working copy is made for cleaning.

In [ ]:
# Load the official UCI PhiUSIIL phishing URL dataset.
# UCI label convention: 1 = legitimate, 0 = phishing.
!pip -q install ucimlrepo

import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo

dataset = fetch_ucirepo(id=967)
df = pd.concat([dataset.data.features, dataset.data.targets], axis=1)
df.columns = [str(c).strip() for c in df.columns]
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Label column candidates:", [c for c in df.columns if c.lower() == "label"])
display(df.head())

df_raw = df.copy()
clean_df = df.copy()
label_col = next(c for c in clean_df.columns if c.lower() == "label")
url_col = next(c for c in clean_df.columns if c.lower() == "url")
print("Original shape:", clean_df.shape)
print("Missing values by column:")
display(clean_df.isna().sum()[clean_df.isna().sum() > 0])

## 2. Missing-value decisions
For every column with missing values, use a suitable strategy and print the decision. If the dataset contains no missing values, record that no imputation is required rather than inventing missing data.

In [ ]:
missing_cols = clean_df.columns[clean_df.isna().any()].tolist()

if not missing_cols:
    print("No missing values were found. No imputation was necessary.")
else:
    for col in missing_cols:
        if pd.api.types.is_numeric_dtype(clean_df[col]):
            clean_df[col] = clean_df[col].fillna(clean_df[col].median())
            print(f"{col}: filled missing values with the median because it is robust to skew and outliers.")
        else:
            clean_df[col] = clean_df[col].fillna("unknown")
            print(f"{col}: filled missing values with 'unknown' to preserve rows without inventing a category.")

## 3. Investigate outliers
Inspect URL length using the IQR rule. Extreme URL lengths can be legitimate characteristics of phishing URLs, so they should be investigated rather than automatically deleted.

In [ ]:
url_length_col = next((c for c in clean_df.columns if c.lower() == "urllength"), None)
if url_length_col is None:
    url_length_col = next(c for c in clean_df.select_dtypes(include=np.number).columns if c.lower() != "label")

q1 = clean_df[url_length_col].quantile(0.25)
q3 = clean_df[url_length_col].quantile(0.75)
iqr = q3 - q1
lower_bound, upper_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = clean_df[(clean_df[url_length_col] < lower_bound) | (clean_df[url_length_col] > upper_bound)]

print("Numeric column inspected:", url_length_col)
print("IQR lower bound:", lower_bound)
print("IQR upper bound:", upper_bound)
print("Outlier count:", len(outliers))
print("Decision: retain plausible extreme URL lengths; unusually long URLs may be a meaningful phishing signal. Investigate malformed values separately rather than deleting all statistical outliers.")

## 4. Correct data types
Ensure the URL field is text and the target label is numeric. Display the corrected types.

In [ ]:
clean_df[url_col] = clean_df[url_col].astype("string")
clean_df[label_col] = pd.to_numeric(clean_df[label_col], errors="coerce")
print(clean_df[[url_col, label_col]].dtypes)
print("Remaining missing labels:", clean_df[label_col].isna().sum())

## 5A. Histogram
**Question:** What does the distribution of URL lengths look like?

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.hist(clean_df[url_length_col].dropna(), bins=40, edgecolor="black")
plt.title("Distribution of URL Lengths")
plt.xlabel("URL length (characters)")
plt.ylabel("Number of records")
plt.tight_layout()
plt.show()

## 5B. Bar chart of category counts
**Question:** How many legitimate and phishing records are present? UCI labels 1 as legitimate and 0 as phishing.

In [ ]:
label_names = clean_df[label_col].map({1: "Legitimate", 0: "Phishing"}).fillna("Other")
counts = label_names.value_counts()
plt.figure(figsize=(7, 5))
plt.bar(counts.index.astype(str), counts.values)
plt.title("Legitimate vs Phishing URL Counts")
plt.xlabel("URL category")
plt.ylabel("Number of records")
plt.tight_layout()
plt.show()

## 5C. Scatter plot
**Question:** Is URL length associated with domain length?

In [ ]:
domain_length_col = next((c for c in clean_df.columns if c.lower() == "domainlength"), None)
if domain_length_col is None:
    domain_length_col = [c for c in clean_df.select_dtypes(include=np.number).columns if c.lower() != "label"][1]
plt.figure(figsize=(8, 5))
plt.scatter(clean_df[url_length_col], clean_df[domain_length_col], alpha=0.25)
plt.title("URL Length vs Domain Length")
plt.xlabel("URL length")
plt.ylabel("Domain length")
plt.tight_layout()
plt.show()

## 5D. Second bar chart (dataset has no date field used here)
**Question:** How does average URL length compare between legitimate and phishing categories?

In [ ]:
group_means = clean_df.assign(category=label_names).groupby("category")[url_length_col].mean()
plt.figure(figsize=(7, 5))
plt.bar(group_means.index.astype(str), group_means.values)
plt.title("Average URL Length by Category")
plt.xlabel("URL category")
plt.ylabel("Average URL length")
plt.tight_layout()
plt.show()

## 5E. Your own chart – box plot
**Question:** Do URL-length distributions differ between legitimate and phishing records?

In [ ]:
plot_data = [
    clean_df.loc[label_names == "Legitimate", url_length_col].dropna(),
    clean_df.loc[label_names == "Phishing", url_length_col].dropna()
]
plt.figure(figsize=(7, 5))
plt.boxplot(plot_data, labels=["Legitimate", "Phishing"], showfliers=False)
plt.title("URL Length Distribution by Category")
plt.xlabel("URL category")
plt.ylabel("URL length")
plt.tight_layout()
plt.show()

## 6. Five findings generated from the data
The statements below are calculated from the loaded dataset; review the printed values and explain them in your own words.

In [ ]:
print("Finding 1 – Dataset size:", clean_df.shape)
print("Finding 2 – Class counts:")
display(label_names.value_counts().to_frame("records"))
print("Finding 3 – URL length summary:")
display(clean_df[url_length_col].describe().to_frame("URL_length"))
print("Finding 4 – Average URL length by label:")
display(clean_df.assign(category=label_names).groupby("category")[url_length_col].mean().to_frame("average_URL_length"))
print("Finding 5 – Missing values remaining:", int(clean_df.isna().sum().sum()))

## 7. Ethics and bias
**Who is represented and who is missing?** The dataset represents URL and webpage observations collected for phishing research. It does not describe the people who receive the links, their age, language, disability, location, or digital literacy; those user groups are therefore not directly represented.

**Who could be treated unfairly?** A detector may incorrectly flag legitimate websites, including small organisations, new domains, or sites using less common URL patterns. False negatives may leave people exposed to malicious sites.

**What should not be used just because it improves accuracy?** Unique identifiers such as `FILENAME` should not be model features because they can cause memorisation without meaningful generalisation. Domain or title information should also be assessed for privacy, proxy effects, and leakage before use. A prediction should support human review, not automatically block or accuse a person or organisation.

## Completion checklist
- [ ] Every column with missing values has a documented decision and reason (or explicitly record none exist).
- [ ] Outliers in a numeric column are counted, investigated and a decision recorded.
- [ ] Incorrect types are corrected.
- [ ] Five charts each have a question, title and labelled axes.
- [ ] Five findings are calculated from the dataset.
- [ ] All three ethics questions are answered specifically.